# Water Crystal classification

Much of the earth’s surface is covered by water. As was pointed out in the 2020 edition of the World Water Development Report, climate change challenges the sustainability of global water resources, so it is important to monitor the quality of water to preserve sustainable water resources. Quality of water can be related to the structure of water crystal, the solid-state of water, so methods to understand water crystals can help to improve water quality. As a first step, a water crystal exploratory analysis has been initiated with the cooperation with the Emoto Peace Project (EPP). The 5K EPP dataset has been created as the first world-wide small dataset of water crystals. Our research focused on reducing the inherent limitations when fitting machine learning models to the 5K EPP dataset. One major result is the classification of water crystals and how to split our small dataset into several related groups. Using the 5K EPP dataset of human observations and past research on snow crystal classification, we created a simple set of visual labels to identify water crystal shapes, in 13 categories. A deep learning-based method has been used to automatically do the classification task with a subset of the label dataset. The classification achieved high accuracy when using a fine-tuning technique.

## Dataset

The 5K EPP Dataset includes 5007 photos of water crystaks classified in 13 categories. This dataset was created under the leaderhip of Prof. Masaru Emoto.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
import datasets.epp_dataset
import matplotlib.pyplot as plt
import matplotlib as mpl
import datetime
from collections import defaultdict
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.manifold import TSNE
import os

# Set the log level to only display errors
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

(train_ds, test_ds, validation_ds), metadata = tfds.load(
    'epp',
    split=['train', 'test', 'validation'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

num_classes = metadata.features['label'].num_classes

print(f"Number of classes: {num_classes}")
print(f"Class names: {metadata.features['label'].names}")

## Image example

In [ ]:
get_label_name = metadata.features['label'].int2str

plt.figure(figsize=(10, 8))

i = 0
for image, label in train_ds.take(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(get_label_name(label))
    plt.axis("off")

    i += 1

## Visualise the distribution

In [ ]:
label_names = metadata.features['label'].names

# Function to extract labels and count their occurrences
def get_label_distribution(dataset):
    label_counts = defaultdict(int)
    for _, label in dataset:
        label_counts[get_label_name(label)] += 1

    return label_counts

# Get label distributions for training and test datasets
train_label_distribution = get_label_distribution(train_ds)
test_label_distribution = get_label_distribution(test_ds)
validation_label_distribution = get_label_distribution(validation_ds)

# Sort the distributions by label ID
train_label_distribution = dict(sorted(train_label_distribution.items()))
test_label_distribution = dict(sorted(test_label_distribution.items()))
validation_label_distribution = dict(sorted(validation_label_distribution.items()))

# Plotting the label distributions
fig, ax = plt.subplots(1, 3, figsize=(12, 6))

ax[0].bar(train_label_distribution.keys(), train_label_distribution.values())
ax[0].set_title('Training Set Label Distribution')
ax[0].set_xlabel('Labels')
ax[0].set_ylabel('Count')
ax[0].tick_params(axis='x', rotation=90)

ax[1].bar(test_label_distribution.keys(), test_label_distribution.values())
ax[1].set_title('Test Set Label Distribution')
ax[1].set_xlabel('Labels')
ax[1].set_ylabel('Count')
ax[1].tick_params(axis='x', rotation=90)

ax[2].bar(validation_label_distribution.keys(), validation_label_distribution.values())
ax[2].set_title('Validation Set Label Distribution')
ax[2].set_xlabel('Labels')
ax[2].set_ylabel('Count')
ax[2].tick_params(axis='x', rotation=90)

plt.show()

## Preprocess dataset

In [ ]:
def normalize_label(image, label):
  return image, tf.one_hot(label, num_classes)

train_ds = train_ds.map(normalize_label, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.cache()
train_ds = train_ds.shuffle(metadata.splits['train'].num_examples)
train_ds = train_ds.batch(32)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.map(normalize_label, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(32)
test_ds = test_ds.cache()
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

validation_ds = validation_ds.map(normalize_label, num_parallel_calls=tf.data.AUTOTUNE)
validation_ds = validation_ds.batch(32)
validation_ds = validation_ds.cache()
validation_ds = validation_ds.prefetch(tf.data.AUTOTUNE)

## Create and execute the model (without active learning)

In [ ]:
# Get label distributions for training and test datasets
label_counts = defaultdict(int)
for _, label in train_ds.unbatch():
    label_counts[np.argmax(label)] += 1

# Calculate class weights
# Inverse frequency ("balanced"), capped so classes with 1-2 training
# images do not dominate the loss.
MAX_CLASS_WEIGHT = 10.0
total_count = sum(label_counts.values())
class_weight = {
    int(k): min(total_count / (num_classes * v), MAX_CLASS_WEIGHT)
    for k, v in label_counts.items()
}

data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip(),
  tf.keras.layers.RandomRotation(0.025),
  tf.keras.layers.RandomTranslation(0.025, 0.025),
  tf.keras.layers.RandomZoom(0.025)
])

preprocess_input = tf.keras.applications.inception_resnet_v2.preprocess_input

base_model = tf.keras.applications.InceptionResNetV2(
  input_shape=(240, 360, 3),
  include_top=False
)
base_model.trainable = True
print("Number of layers in the base model: ", len(base_model.layers)) # Number of layers: 780
for layer in base_model.layers[:770]:                                 # Freeze: 0 --> 770, Unfreeze: 771 --> 780
  layer.trainable = False

global_max_layer = tf.keras.layers.GlobalMaxPool2D()
prediction_layer = tf.keras.layers.Dense(num_classes, activation="softmax")

inputs = tf.keras.Input(shape=(240, 360, 3))
x = inputs
x = data_augmentation(x)
x = preprocess_input(x)
x = base_model(x)
x = global_max_layer(x)
# Fully Layer 01
x = tf.keras.layers.Dense(300, kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
#x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Activation("relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
# Fully Layer 02
x = tf.keras.layers.Dense(300, kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
#x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Activation("relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
# Fully Layer 03
x = tf.keras.layers.Dense(300, kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
#x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Activation("relu")(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = prediction_layer(x)
#outputs = tf.keras.layers.Concatenate()([outputs, x])
model = tf.keras.Model(inputs, outputs)
model.compile(
  optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
  loss=tf.keras.losses.CategoricalCrossentropy(),
  metrics=["categorical_accuracy"]
)

# Callbacks
log_dir = "logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(
  log_dir=log_dir,
  histogram_freq=1,
  profile_batch = 0
)

checkpoint_filepath = "./checkpoint.weights.h5"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
  checkpoint_filepath,
  save_weights_only=True,
  monitor="val_categorical_accuracy",
  mode="max",
  save_best_only=True
)

backup_callback = tf.keras.callbacks.BackupAndRestore(
  backup_dir="./backup",
  save_freq="epoch",
  delete_checkpoint=True
)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=10,
    restore_best_weights=True,
)

callbacks = [
  tensorboard_callback,
  checkpoint_callback,
  backup_callback,
  early_stopping_callback
]

history = model.fit(
  train_ds,
  epochs = 100,
  validation_data = validation_ds,
  class_weight = class_weight,
  callbacks=callbacks
)

model.save('model.keras')

# Save the class list next to the model, in the order of its output units.
# playground.py reads it, so predictions are always named correctly.
import json
from datasets.epp_dataset.epp_dataset_builder import load_classes
classes = load_classes()
assert [c['code'] for c in classes] == metadata.features['label'].names, (
    'classes.csv changed after the dataset was built; rebuild it first'
)
with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(classes, f, ensure_ascii=False, indent=2)

score = model.evaluate(test_ds)
print("Test Loss: {}".format(score[0]))
print("Test Categorical Accuracy:: {}".format(score[1]))

## Statistics

### Confusion matrix

In [ ]:
def confusion_matrix_plot(model, dataset, title):
    actual = [labels for _, labels in dataset.unbatch()]
    predicted = model.predict(dataset)

    actual = tf.stack(actual, axis=0)
    actual = tf.argmax(actual, axis=1)

    predicted = tf.concat(predicted, axis=0)
    predicted = tf.argmax(predicted, axis=1)

    # Plot the confusion matrix
    cm = confusion_matrix(actual, predicted)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='g', xticklabels=label_names, yticklabels=label_names)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


confusion_matrix_plot(model, test_ds, "Test Dataset Confusion Matrix")
confusion_matrix_plot(model, train_ds, "Train Dataset Confusion Matrix")
confusion_matrix_plot(model, validation_ds, "Validation Dataset Confusion Matrix")

### Distribution

In [ ]:
# Create a new model that outputs the features from the desired layer
feature_extraction_model = tf.keras.Model(inputs=model.input, outputs=global_max_layer.output)

# Extract features from the dataset
features = []
labels = []

for images, label in train_ds.unbatch():
    features.append(feature_extraction_model.predict(np.expand_dims(images, axis=0)))
    labels.append(np.argmax(label))

features = np.squeeze(np.array(features))
labels = np.array(labels)

# Apply t-SNE
tsne = TSNE(n_components=3, random_state=42)
tsne_features = tsne.fit_transform(features)

# Plot the t-SNE features
num_classes = metadata.features['label'].num_classes
bounds = np.linspace(0, num_classes, num_classes + 1)
cmap = plt.cm.jet
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)
get_label_name = metadata.features['label'].int2str

fig = plt.figure(figsize=(12, 10))

ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(tsne_features[:, 0], tsne_features[:, 1], tsne_features[:, 2], c=labels, cmap=cmap, alpha=0.7, norm=norm, s=15)
ax.legend(handles=scatter.legend_elements()[0], labels=[get_label_name(i) for i in range(num_classes)], title="Labels", loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xlabel('t-SNE Feature 1')
ax.set_ylabel('t-SNE Feature 2')
ax.set_zlabel('t-SNE Feature 3')

plt.show()